## Setup
This notebook requires a DeepSeek API key (free tier works).
Get one at: https://platform.deepseek.com
If running locally, add DEEPSEEK_API_KEY to a .env file.
If running on Colab, you will be prompted to enter it.

In [ ]:
import sys
!{sys.executable} -m pip install orchestral-ai pennylane torch torchvision python-dotenv

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from orchestral import Agent, define_tool
from orchestral.llm import VLLM
from dotenv import load_dotenv
import numpy as np
import os

load_dotenv()

api_key = os.getenv('DEEPSEEK_API_KEY')
if not api_key:
    import getpass
    api_key = getpass.getpass("Enter DeepSeek API key: ")

In [ ]:
@define_tool()
def hilbert_space_dimension(n_qubits: int) -> str:
    """Returns the dimension of the Hilbert space for an n-qubit system.

    Args:
        n_qubits: Number of qubits in the system
    Returns:
        Dimension of the Hilbert space (2^n_qubits) as a string
    """
    dim = 2 ** n_qubits
    return f"A {n_qubits}-qubit system has a Hilbert space of dimension {dim}"

agent = Agent(
    llm=VLLM(model='deepseek-chat', host='https://api.deepseek.com', api_key=api_key),
    tools=[hilbert_space_dimension],
    system_prompt="You are a quantum computing assistant. Use tools when asked about quantum systems."
)

response = agent.run("What is the dimension of the Hilbert space for a 4-qubit system?")
print(response.text)

In [ ]:
def build_circuit(n_qubits, n_layers, entanglement_type, embedding_type):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def circuit(inputs, weights):
        if embedding_type == "angle":
            qml.AngleEmbedding(inputs, wires=range(n_qubits))
        elif embedding_type == "amplitude":
            qml.AmplitudeEmbedding(inputs, wires=range(n_qubits), normalize=True)
        for _ in range(n_layers):
            for i in range(n_qubits):
                qml.RY(weights[_, i], wires=i)
            if entanglement_type == "linear":
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i+1])
            elif entanglement_type == "circular":
                for i in range(n_qubits):
                    qml.CNOT(wires=[i, (i+1) % n_qubits])
            elif entanglement_type == "full":
                for i in range(n_qubits):
                    for j in range(i+1, n_qubits):
                        qml.CNOT(wires=[i, j])
        return qml.expval(qml.PauliZ(0))

    return circuit

In [ ]:
class HybridQNN(nn.Module):
    def __init__(self, n_qubits, n_layers, entanglement_type, embedding_type):
        super().__init__()
        self.pre    = nn.Linear(784, n_qubits)
        self.weights = nn.Parameter(torch.randn(n_layers, n_qubits))
        self.post   = nn.Linear(1, 10)
        self.circuit = build_circuit(n_qubits, n_layers, entanglement_type, embedding_type)

    def forward(self, x):
        x = torch.sigmoid(self.pre(x.view(-1, 784)))
        x = torch.stack([self.circuit(x[i], self.weights)
                         for i in range(len(x))]).float()
        return self.post(x.unsqueeze(1))

In [ ]:
transform    = transforms.Compose([transforms.ToTensor()])
full_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
subset       = Subset(full_dataset, range(512))
loader       = DataLoader(subset, batch_size=32, shuffle=True)

In [ ]:
@define_tool()
def train_qnn(epochs: int, learning_rate: float) -> str:
    """Trains a hybrid quantum-classical neural network on MNIST.

    Args:
        epochs: Number of training epochs
        learning_rate: Learning rate for Adam optimizer
    Returns:
        Training loss and accuracy
    """
    model     = HybridQNN(4, 2, "linear", "angle")
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss, correct, total = 0, 0, 0
        for X, y in loader:
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (out.argmax(1) == y).sum().item()
            total      += len(y)

    acc = correct / total
    return f"loss={round(total_loss/len(loader), 4)}, accuracy={round(acc, 4)}"

In [ ]:
agent3 = Agent(
    llm=VLLM(model='deepseek-chat', host='https://api.deepseek.com', api_key=api_key),
    tools=[train_qnn],
    system_prompt="""You are a quantum circuit optimization assistant.
Find the optimal learning rate for the hybrid QNN by trying at least 5 different
learning rates using epochs=3. Report the best learning rate found."""
)

response = agent3.run(
    "Find the optimal learning rate for the hybrid QNN. Try at least 5 values.",
    max_iterations=15
)
print(response.text)